# Persona Compound Fairness — Results (AV + HV)

Comparing triage outcomes across five personae compounds (informed_advocate, anxious_patient, limited_communicator, dismissive_patient, default).

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# --- Configuration ---
RUN_ID = "persona_20260521_172943_d0326dbc"
FIG_SUFFIX = "_gpt54mini"
RESULTS_DIR = Path("../results/runs") / RUN_ID / "conversations"

TRIAGE_ORD = {"self-care": 0, "non-urgent": 1, "urgent": 2}
MAX_ORD = max(TRIAGE_ORD.values())

# --- Load results (AV + HV only) ---
results = []
for p in sorted(RESULTS_DIR.glob("*.json")):
    with open(p) as f:
        r = json.load(f)
    if r.get("dataset_name") not in ("av", "hv"):
        continue
    assessments = r.get("assessments", [])
    if assessments:
        votes = [a["urgency"] for a in assessments if a.get("urgency")]
        if votes:
            r["predicted_triage"] = Counter(votes).most_common(1)[0][0]
    results.append(r)

df = pd.DataFrame(results)
df = df[df["predicted_triage"].notna() & df["gold_triage"].notna()].copy()
df["gold_ord"] = df["gold_triage"].map(TRIAGE_ORD)
df["pred_ord"] = df["predicted_triage"].map(TRIAGE_ORD)
df = df[df["pred_ord"].notna() & df["gold_ord"].notna()].copy()

print(f"Loaded {len(df)} valid results")
for ds, cnt in sorted(df.groupby("dataset_name").size().items()):
    print(f"  {ds}: {cnt}")
print(f"\nConditions: {sorted(df['condition_name'].unique())}")
print(f"Cases per condition: {df.groupby('condition_name').size().to_dict()}")

In [ ]:
# --- Classify and compute metrics ---
def classify(row):
    if row.pred_ord == row.gold_ord:
        return "correct"
    return "over" if row.pred_ord > row.gold_ord else "under"

df["direction"] = df.apply(classify, axis=1)

print(f"Direction distribution:\n{df['direction'].value_counts()}\n")


def bootstrap_rate(values, n_boot=10_000, seed=42):
    arr = np.array(values, dtype=float)
    rng = np.random.default_rng(seed)
    boots = [rng.choice(arr, size=len(arr), replace=True).mean() for _ in range(n_boot)]
    return arr.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5)


def compute_metrics(grp_df):
    metrics = []
    for cond, grp in grp_df.groupby("condition_name"):
        ot_eligible = grp[grp["gold_triage"] != "urgent"]
        over_rate = (ot_eligible["direction"] == "over").mean() if len(ot_eligible) else 0

        ut_eligible = grp[grp["gold_triage"] != "self-care"]
        under_rate = (ut_eligible["direction"] == "under").mean() if len(ut_eligible) else 0

        metrics.append({
            "condition": cond,
            "over_rate": over_rate,
            "under_rate": under_rate,
            "n": len(grp),
        })
    return pd.DataFrame(metrics)


PERSONA_FULL = {
    "anxious_patient": "Anxious Patient",
    "baseline": "Default",
    "dismissive_patient": "Dismissive Patient",
    "informed_advocate": "Informed Advocate",
    "limited_communicator": "Limited Communicator",
}

mdf = compute_metrics(df)
summary = mdf.sort_values("over_rate", ascending=True).reset_index(drop=True)
summary["over_triage"] = summary["over_rate"].map(lambda x: f"{x:.1%}")
summary["under_triage"] = summary["under_rate"].map(lambda x: f"{x:.1%}")
print("=== AV + HV combined ===")
display(summary[["condition", "over_triage", "under_triage", "n"]])

In [ ]:
# --- Data: pairwise triage changes and differences ---

personas = sorted(df["condition_name"].unique())
pairs = [(i, j) for i in range(len(personas)) for j in range(i + 1, len(personas))]

piv_pred = df.pivot_table(
    index=["case_id", "repetition"], columns="condition_name",
    values="pred_ord", aggfunc="mean",
).dropna()

rng_cr = np.random.default_rng(42)

def _boot_rate_cr(arr, rng, n_boot=10_000):
    arr = np.array(arr, dtype=float)
    boots = [rng.choice(arr, size=len(arr), replace=True).mean() for _ in range(n_boot)]
    return arr.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5)

change_results = []
for i, j in pairs:
    p1, p2 = personas[i], personas[j]
    diff = piv_pred[p2].values - piv_pred[p1].values

    any_change = (diff != 0).astype(float)
    goes_higher = (diff > 0).astype(float)
    goes_lower = (diff < 0).astype(float)
    stays_same = (diff == 0).astype(float)

    m_any, lo_any, hi_any = _boot_rate_cr(any_change, rng_cr)
    m_hi, lo_hi, hi_hi = _boot_rate_cr(goes_higher, rng_cr)
    m_lo, lo_lo, hi_lo = _boot_rate_cr(goes_lower, rng_cr)
    m_same, _, _ = _boot_rate_cr(stays_same, rng_cr)

    change_results.append({
        "p1": PERSONA_FULL[p1], "p2": PERSONA_FULL[p2],
        "label": f"({PERSONA_FULL[p1]}, {PERSONA_FULL[p2]})",
        "any_mean": m_any, "any_lo": lo_any, "any_hi": hi_any,
        "higher_mean": m_hi, "higher_lo": lo_hi, "higher_hi": hi_hi,
        "lower_mean": m_lo, "lower_lo": lo_lo, "lower_hi": hi_lo,
        "same_mean": m_same,
        "n": len(diff),
    })

cr_df = pd.DataFrame(change_results)
cr_fly = cr_df.sort_values("any_mean", ascending=True).reset_index(drop=True)

# Shared pair ordering for all pairwise plots
ordered_pairs = [
    ("anxious_patient", "baseline"),
    ("dismissive_patient", "baseline"),
    ("informed_advocate", "baseline"),
    ("limited_communicator", "baseline"),
    ("anxious_patient", "informed_advocate"),
    ("dismissive_patient", "informed_advocate"),
    ("limited_communicator", "informed_advocate"),
    ("anxious_patient", "limited_communicator"),
    ("dismissive_patient", "limited_communicator"),
    ("anxious_patient", "dismissive_patient"),
]
group_starts = [0, 4, 7, 9]

pair_names = [(PERSONA_FULL[p1], PERSONA_FULL[p2]) for p1, p2 in ordered_pairs]
n_p = len(ordered_pairs)
y_positions = []
current_y = 0
for k in range(n_p):
    if k > 0 and k in group_starts:
        current_y += 0.4
    y_positions.append(current_y)
    current_y += 1
y = np.array(y_positions)
title_y = y[-1] + 0.8

# Pairwise over/under triage rate differences (paired bootstrap)
metric_defs = [
    ("Over-triage Rate Difference across Pairs",
     lambda d: d[d["gold_triage"] != "urgent"],
     lambda d: (d["direction"] == "over").astype(float)),
    ("Under-triage Rate Difference across Pairs",
     lambda d: d[d["gold_triage"] != "self-care"],
     lambda d: (d["direction"] == "under").astype(float)),
]

all_results = {}
for name, filt_fn, val_fn in metric_defs:
    rng = np.random.default_rng(42)
    sub = filt_fn(df).copy()
    sub = sub.assign(_v=val_fn(sub))
    piv = sub.pivot_table(
        index=["case_id", "repetition"], columns="condition_name",
        values="_v", aggfunc="mean",
    ).dropna()

    results = []
    for i, j in pairs:
        d = (piv[personas[i]] - piv[personas[j]]).values
        boots = np.array([
            rng.choice(d, size=len(d), replace=True).mean() for _ in range(10_000)
        ])
        lo, hi = np.percentile(boots, [2.5, 97.5])
        label = f"({PERSONA_FULL[personas[i]]}, {PERSONA_FULL[personas[j]]})"
        results.append({"label": label, "mean": d.mean(), "lo": lo, "hi": hi})
    all_results[name] = pd.DataFrame(results)

print(f"Computed {len(change_results)} pairwise change rates")
print(f"Computed pairwise differences for: {list(all_results.keys())}")

In [ ]:
# --- Pairwise change rate: how often does the model decision change? ---
# Same ordering as the pairwise differences butterfly.
# Orange = triage changed. Bars show % out of all patients.

# Build change data in ordered_pairs order
change_lookup = {}
for _, r in cr_df.iterrows():
    p1_key = [k for k, v in PERSONA_FULL.items() if v == r.p1][0]
    p2_key = [k for k, v in PERSONA_FULL.items() if v == r.p2][0]
    change_lookup[(p1_key, p2_key)] = r.any_mean * 100
    change_lookup[(p2_key, p1_key)] = r.any_mean * 100

change_pcts = np.array([change_lookup[(p1, p2)] for p1, p2 in ordered_pairs])

fig, ax = plt.subplots(figsize=(14, 8))

ax.barh(y, change_pcts, color="#e67e22", edgecolor="none", height=0.6, zorder=3)

# Annotations (not bold), just right of bar
for k in range(n_p):
    ax.text(change_pcts[k] + 0.3, y[k], f"{change_pcts[k]:.1f}%",
            va="center", ha="left", fontsize=9, color="black", zorder=5)

ax.set_xlim(0, max(change_pcts) * 1.4)

# Y-axis: use two text columns via annotations, no tick labels
ax.set_yticks(list(y) + [title_y])
ax.set_yticklabels([""] * (n_p + 1))
ax.set_ylim(-0.5, title_y + 0.5)

# Draw persona names as two aligned text columns (P1 leftmost, P2 right)
fig.canvas.draw()
for k in range(n_p):
    ax.text(-0.22, y[k], pair_names[k][0], va="center", ha="right",
            fontsize=10, transform=ax.get_yaxis_transform())
    ax.text(-0.02, y[k], pair_names[k][1], va="center", ha="right",
            fontsize=10, transform=ax.get_yaxis_transform())

# Header row
ax.text(-0.22, title_y, "Persona 1", va="center", ha="right",
        fontsize=10, fontweight="bold", transform=ax.get_yaxis_transform())
ax.text(-0.02, title_y, "Persona 2", va="center", ha="right",
        fontsize=10, fontweight="bold", transform=ax.get_yaxis_transform())

ax.yaxis.set_tick_params(length=0)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.xaxis.set_major_formatter(
    plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax.tick_params(axis="x", labelsize=9)

# Row separators
xlim = ax.get_xlim()
for k in range(n_p - 1):
    mid = (y[k] + y[k + 1]) / 2
    is_group_boundary = (k + 1) in group_starts
    if is_group_boundary:
        ax.hlines(mid, xlim[0], xlim[1], color="grey", alpha=0.5,
                  lw=1.0, zorder=1)
    else:
        ax.hlines(mid, xlim[0], xlim[1], color="grey", alpha=0.2,
                  lw=0.5, zorder=0)

ax.set_xlabel("Patients whose triage changed (%)", fontsize=10)
ax.set_title("Same patient, different persona: does the model decision change?",
             fontsize=13, pad=15)

plt.tight_layout()
plt.savefig(f"../figures/change_rate_stacked{FIG_SUFFIX}.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Butterfly — who gets triaged higher? ---
# Uses ordered_pairs ordering (same as all other pairwise plots).

# Build lookup from cr_df for higher/lower rates by persona key pair
cr_lookup = {}
for _, r in cr_df.iterrows():
    p1_key = [k for k, v in PERSONA_FULL.items() if v == r.p1][0]
    p2_key = [k for k, v in PERSONA_FULL.items() if v == r.p2][0]
    cr_lookup[(p1_key, p2_key)] = r
    cr_lookup[(p2_key, p1_key)] = r

# For each ordered pair, "P2 triaged higher" = P2 got a higher triage than P1
p2_higher = np.zeros(n_p)
p2_hi_lo = np.zeros(n_p)
p2_hi_hi = np.zeros(n_p)
p1_higher = np.zeros(n_p)
p1_hi_lo = np.zeros(n_p)
p1_hi_hi = np.zeros(n_p)

for k, (p1_key, p2_key) in enumerate(ordered_pairs):
    r = cr_lookup[(p1_key, p2_key)]
    r_p1 = [key for key, v in PERSONA_FULL.items() if v == r.p1][0]
    if r_p1 == p1_key:
        p2_higher[k] = r.higher_mean * 100
        p2_hi_lo[k] = r.higher_lo * 100
        p2_hi_hi[k] = r.higher_hi * 100
        p1_higher[k] = r.lower_mean * 100
        p1_hi_lo[k] = r.lower_lo * 100
        p1_hi_hi[k] = r.lower_hi * 100
    else:
        p1_higher[k] = r.higher_mean * 100
        p1_hi_lo[k] = r.higher_lo * 100
        p1_hi_hi[k] = r.higher_hi * 100
        p2_higher[k] = r.lower_mean * 100
        p2_hi_lo[k] = r.lower_lo * 100
        p2_hi_hi[k] = r.lower_hi * 100

fig, ax = plt.subplots(figsize=(14, 8))

ax.barh(y, p2_higher, color="#e74c3c", edgecolor="none", height=0.6, zorder=3)
ax.barh(y, -p1_higher, color="#3498db", edgecolor="none", height=0.6, zorder=3)

ax.errorbar(p2_higher, y, xerr=[p2_higher - p2_hi_lo, p2_hi_hi - p2_higher],
            fmt="none", ecolor="black", elinewidth=0.8, capsize=2, zorder=4)
ax.errorbar(-p1_higher, y, xerr=[p1_higher - p1_hi_lo, p1_hi_hi - p1_higher],
            fmt="none", ecolor="black", elinewidth=0.8, capsize=2, zorder=4)

ax.axvline(0, color="white", linewidth=2, zorder=5)

for k in range(n_p):
    ax.text(-0.3, y[k], f"{p1_higher[k]:.1f}%",
            va="center", ha="right", fontsize=8, color="white",
            fontweight="bold", zorder=6)
    ax.text(0.3, y[k], f"{p2_higher[k]:.1f}%",
            va="center", ha="left", fontsize=8, color="white",
            fontweight="bold", zorder=6)

# Y-axis: two-column text annotations (same approach as change rate plot)
ax.set_yticks(list(y) + [title_y])
ax.set_yticklabels([""] * (n_p + 1))
ax.set_ylim(-0.5, title_y + 0.5)

fig.canvas.draw()
for k in range(n_p):
    ax.text(-0.22, y[k], pair_names[k][0], va="center", ha="right",
            fontsize=10, transform=ax.get_yaxis_transform())
    ax.text(-0.02, y[k], pair_names[k][1], va="center", ha="right",
            fontsize=10, transform=ax.get_yaxis_transform())

ax.text(-0.22, title_y, "Persona 1", va="center", ha="right",
        fontsize=10, fontweight="bold", transform=ax.get_yaxis_transform())
ax.text(-0.02, title_y, "Persona 2", va="center", ha="right",
        fontsize=10, fontweight="bold", transform=ax.get_yaxis_transform())

ax.yaxis.set_tick_params(length=0)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.xaxis.set_major_formatter(
    plt.FuncFormatter(lambda v, _: f"{abs(v):.0f}%"))

# Row separators
xlim = ax.get_xlim()
for k in range(n_p - 1):
    mid = (y[k] + y[k + 1]) / 2
    is_group_boundary = (k + 1) in group_starts
    if is_group_boundary:
        ax.hlines(mid, xlim[0], xlim[1], color="grey", alpha=0.5,
                  lw=1.0, zorder=1)
    else:
        ax.hlines(mid, xlim[0], xlim[1], color="grey", alpha=0.2,
                  lw=0.5, zorder=0)

# Directional labels just above the Persona 1/2 header
center_frac = (0 - xlim[0]) / (xlim[1] - xlim[0])
ax.text(center_frac - 0.01, 1.005, "← Persona 1 triaged higher",
        transform=ax.transAxes, fontsize=10, ha="right", color="#3498db")
ax.text(center_frac + 0.01, 1.005, "Persona 2 triaged higher →",
        transform=ax.transAxes, fontsize=10, ha="left", color="#e74c3c")

ax.set_xlabel("Patients whose triage goes higher (%) in comparison to counterpart", fontsize=10)
ax.set_title("Same patient, different persona: who gets triaged higher?",
             fontsize=13, pad=25)

plt.tight_layout()
plt.savefig(f"../figures/change_rate_butterfly{FIG_SUFFIX}.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Change rate summary table ---
# Uses ordered_pairs ordering (same as butterfly and all pairwise plots).

table_rows = []
for k, (p1_key, p2_key) in enumerate(ordered_pairs):
    r = cr_lookup[(p1_key, p2_key)]
    table_rows.append({
        "Persona 1": pair_names[k][0],
        "Persona 2": pair_names[k][1],
        "P1 triaged higher (95% CI)": f"{p1_higher[k]:.1f}% [{p1_hi_lo[k]:.1f}, {p1_hi_hi[k]:.1f}]",
        "P2 triaged higher (95% CI)": f"{p2_higher[k]:.1f}% [{p2_hi_lo[k]:.1f}, {p2_hi_hi[k]:.1f}]",
        "Any change (95% CI)": f"{r.any_mean*100:.1f}% [{r.any_lo*100:.1f}, {r.any_hi*100:.1f}]",
        "Same": f"{r.same_mean*100:.1f}%",
    })
display(pd.DataFrame(table_rows))

In [ ]:
# --- Risk Ratio: who is more likely to be triaged higher? ---
# RR = P(P2 triaged higher) / P(P1 triaged higher).
# If RR > 1 → P2 more likely → plot right (red), show RR as-is.
# If RR < 1 → P1 more likely → plot left (blue), show 1/RR (so always > 1).
# x-axis is log₂ scale, symmetric, with tick labels always > 1.
# Grey (non-significant) CIs extend naturally across the center line.

rng_rr = np.random.default_rng(42)

rr_lookup = {}
for _, r in cr_fly.iterrows():
    p1_key = [k for k, v in PERSONA_FULL.items() if v == r.p1][0]
    p2_key = [k for k, v in PERSONA_FULL.items() if v == r.p2][0]
    diff = piv_pred[p2_key].values - piv_pred[p1_key].values
    p1_higher = (diff < 0).astype(float)
    p2_higher = (diff > 0).astype(float)

    log_rrs = []
    for _ in range(10_000):
        idx = rng_rr.integers(0, len(diff), size=len(diff))
        b_p1 = p1_higher[idx].mean()
        b_p2 = p2_higher[idx].mean()
        if b_p1 > 0 and b_p2 > 0:
            log_rrs.append(np.log(b_p2 / b_p1))
    log_rrs = np.array(log_rrs)

    rr_lookup[(p1_key, p2_key)] = {
        "rr": np.exp(log_rrs.mean()),
        "rr_lo": np.exp(np.percentile(log_rrs, 2.5)),
        "rr_hi": np.exp(np.percentile(log_rrs, 97.5)),
    }

# Build RR data in the same fixed order as pairwise differences
rr_data = []
for p1_key, p2_key in ordered_pairs:
    if (p1_key, p2_key) in rr_lookup:
        d = dict(rr_lookup[(p1_key, p2_key)])
    else:
        d = rr_lookup[(p2_key, p1_key)]
        d = {"rr": 1.0 / d["rr"], "rr_lo": 1.0 / d["rr_hi"], "rr_hi": 1.0 / d["rr_lo"]}
    rr_data.append(d)

# Debug: raw values with high precision
print("Raw RR (P2/P1) and displayed values:")
for k, (p1_key, p2_key) in enumerate(ordered_pairs):
    r = rr_data[k]
    if r["rr"] >= 1:
        disp_rr, disp_lo, disp_hi = r["rr"], r["rr_lo"], r["rr_hi"]
        side = "RIGHT (red)"
    else:
        disp_rr = 1/r["rr"]
        disp_lo = 1/r["rr_hi"]
        disp_hi = 1/r["rr_lo"]
        side = "LEFT (blue)"
    print(f"  {PERSONA_FULL[p1_key]:25s} vs {PERSONA_FULL[p2_key]:25s}: "
          f"raw={r['rr']:.6f} [{r['rr_lo']:.6f}, {r['rr_hi']:.6f}] "
          f"→ display={disp_rr:.6f} [{disp_lo:.6f}, {disp_hi:.6f}] {side}")

# Compute dynamic xlim from data
all_log2 = []
for r in rr_data:
    all_log2.extend([np.log2(r["rr_lo"]), np.log2(r["rr_hi"])])
max_log = max(abs(v) for v in all_log2) * 1.1

# Forest plot (reuses y, pair_names, group_starts, title_y from pairwise cell)
fig, ax = plt.subplots(figsize=(14, 8))

cap_h = 0.15

for k in range(n_p):
    r = rr_data[k]
    raw_rr = r["rr"]
    raw_lo = r["rr_lo"]
    raw_hi = r["rr_hi"]

    if raw_rr >= 1:
        color = "#e74c3c" if raw_lo > 1 else "#555555"
        display_rr = raw_rr
        display_lo = raw_lo
        display_hi = raw_hi
    else:
        color = "#3498db" if raw_hi < 1 else "#555555"
        display_rr = 1.0 / raw_rr
        display_lo = 1.0 / raw_hi
        display_hi = 1.0 / raw_lo

    log2_pt = np.log2(raw_rr)
    log2_lo = np.log2(raw_lo)
    log2_hi = np.log2(raw_hi)

    ax.plot([log2_lo, log2_hi], [y[k], y[k]], color=color, lw=2, zorder=3)
    ax.plot([log2_lo, log2_lo], [y[k] - cap_h, y[k] + cap_h],
            color=color, lw=1.5, zorder=3)
    ax.plot([log2_hi, log2_hi], [y[k] - cap_h, y[k] + cap_h],
            color=color, lw=1.5, zorder=3)
    ax.plot(log2_pt, y[k], "o", color=color, markersize=7, zorder=4)

    txt = f"{display_rr:.2f} [{display_lo:.2f}, {display_hi:.2f}]"
    ax.text(log2_pt, y[k] + 0.22, txt, va="bottom", ha="center",
            fontsize=8, color="black", zorder=6)

ax.set_xlim(-max_log, max_log)

ax.axvline(0, color="grey", linewidth=0.8, linestyle="--", zorder=1)

# Row separators
fig.canvas.draw()
xlim = ax.get_xlim()
for k in range(n_p - 1):
    mid = (y[k] + y[k + 1]) / 2
    is_group_boundary = (k + 1) in group_starts
    if is_group_boundary:
        ax.hlines(mid, xlim[0], xlim[1], color="grey", alpha=0.5,
                  lw=1.0, zorder=1)
    else:
        ax.hlines(mid, xlim[0], xlim[1], color="grey", alpha=0.2,
                  lw=0.5, zorder=0)

# Left y-axis: Persona 1
left_labels = [p[0] for p in pair_names] + ["Persona 1"]
left_ticks = list(y) + [title_y]
ax.set_yticks(left_ticks)
ax.set_yticklabels(left_labels, fontsize=10)
ax.get_yticklabels()[-1].set_fontweight("bold")
ax.set_ylim(-0.5, title_y + 0.5)

# Right y-axis: Persona 2
ax2 = ax.twinx()
ax2.set_ylim(ax.get_ylim())
right_labels = [p[1] for p in pair_names] + ["Persona 2"]
ax2.set_yticks(left_ticks)
ax2.set_yticklabels(right_labels, fontsize=10)
ax2.get_yticklabels()[-1].set_fontweight("bold")
ax2.spines[["top", "right", "left"]].set_visible(False)

ax.spines[["top", "right", "left"]].set_visible(False)
ax.yaxis.set_tick_params(length=0)
ax2.yaxis.set_tick_params(length=0)

# x-axis: dynamic ticks, symmetric, always showing values > 1
candidate_ticks = [1.0, 1.5, 2.0, 3.0, 5.0, 10.0, 20.0, 50.0]
max_rr = 2 ** max_log
tick_rr_vals = [v for v in candidate_ticks if v <= max_rr]
all_ticks = [-np.log2(v) for v in reversed(tick_rr_vals[1:])] + [np.log2(v) for v in tick_rr_vals]
all_labels = [str(v) for v in reversed(tick_rr_vals[1:])] + [str(v) for v in tick_rr_vals]
ax.set_xticks(all_ticks)
ax.set_xticklabels(all_labels, fontsize=9)
ax.set_xlabel("Risk Ratio (log scale)", fontsize=10)

# Directional labels
center_frac = (0 - xlim[0]) / (xlim[1] - xlim[0])
ax.text(center_frac - 0.01, 1.005,
        "← RR = P1 triaged higher / P2 triaged higher",
        transform=ax.transAxes, fontsize=10, ha="right", color="#3498db")
ax.text(center_frac + 0.01, 1.005,
        "RR = P2 triaged higher / P1 triaged higher →",
        transform=ax.transAxes, fontsize=10, ha="left", color="#e74c3c")

ax.set_title("Same patient, different persona: risk ratio of being triaged higher when triage differs",
             fontsize=13, pad=40)

plt.tight_layout()
plt.savefig(f"../figures/change_rate_risk_ratio{FIG_SUFFIX}.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Heatmap: pairwise risk ratios (triangular, same layout as OT/UT heatmap) ---

from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

persona_hm_order = [
    "dismissive_patient", "informed_advocate", "baseline",
    "limited_communicator", "anxious_patient",
]
persona_hm_labels = [
    "Dismissive\nPatient", "Informed\nAdvocate", "Default",
    "Limited\nCommunicator", "Anxious\nPatient",
]

row_personas_rr = persona_hm_order[:-1]
col_personas_rr = persona_hm_order[1:][::-1]
row_labels_rr = persona_hm_labels[:-1]
col_labels_rr = persona_hm_labels[1:][::-1]
n_rows_rr = len(row_personas_rr)
n_cols_rr = len(col_personas_rr)

cmap_rr = LinearSegmentedColormap.from_list("BlWtRd", ["#3498db", "#ffffff", "#e74c3c"])
cmap_rr.set_bad("#f5f5f5")

fig, (ax, cax) = plt.subplots(
    1, 2, figsize=(16, 5),
    gridspec_kw={"width_ratios": [1, 0.02], "wspace": 0.02},
)

matrix_rr = np.full((n_rows_rr, n_cols_rr), np.nan)
annot_rr = [[""] * n_cols_rr for _ in range(n_rows_rr)]

for ri, p_row in enumerate(row_personas_rr):
    for ci, p_col in enumerate(col_personas_rr):
        idx_row = persona_hm_order.index(p_row)
        idx_col = persona_hm_order.index(p_col)
        if idx_col <= idx_row:
            continue
        key_fwd = (p_row, p_col)
        key_rev = (p_col, p_row)
        if key_fwd in rr_lookup:
            d = rr_lookup[key_fwd]
        else:
            d_rev = rr_lookup[key_rev]
            d = {"rr": 1.0 / d_rev["rr"], "rr_lo": 1.0 / d_rev["rr_hi"], "rr_hi": 1.0 / d_rev["rr_lo"]}

        log2_rr = np.log2(d["rr"])
        matrix_rr[ri, ci] = log2_rr

        rr_val = d["rr"]
        rr_lo = d["rr_lo"]
        rr_hi = d["rr_hi"]
        sig = "*" if rr_lo > 1 or rr_hi < 1 else ""
        annot_rr[ri][ci] = f"{rr_val:.2f}{sig}\n[{rr_lo:.2f}, {rr_hi:.2f}]"

masked_rr = np.ma.masked_invalid(matrix_rr)
vmax_rr = max(abs(np.nanmin(matrix_rr)), abs(np.nanmax(matrix_rr)))
norm_rr = TwoSlopeNorm(vmin=-vmax_rr, vcenter=0, vmax=vmax_rr)

ax.imshow(masked_rr, cmap=cmap_rr, norm=norm_rr, aspect="auto")

for i in range(n_rows_rr):
    for j in range(n_cols_rr):
        if annot_rr[i][j]:
            ax.text(j, i, annot_rr[i][j], ha="center", va="center",
                    fontsize=10, color="black", linespacing=1.3)

for edge in range(max(n_rows_rr, n_cols_rr) + 1):
    if edge <= n_rows_rr:
        ax.axhline(edge - 0.5, color="white", lw=2)
    if edge <= n_cols_rr:
        ax.axvline(edge - 0.5, color="white", lw=2)

ax.text(-0.5, -0.85, "Persona 2 →\nPersona 1 ↓", ha="right", va="center",
        fontsize=12, color="black", linespacing=1.5,
        transform=ax.transData)

ax.set_xticks(range(n_cols_rr))
ax.set_xticklabels(col_labels_rr, fontsize=12, linespacing=1.1)
ax.xaxis.tick_top()
ax.tick_params(axis="x", pad=12)

ax.set_yticks(range(n_rows_rr))
ax.set_yticklabels(row_labels_rr, fontsize=12, linespacing=1.1)
ax.tick_params(axis="y", pad=12)

ax.tick_params(length=0)
ax.spines[["top", "right", "bottom", "left"]].set_visible(False)

ax.set_title(
    "Risk Ratio of being triaged higher (P2 higher / P1 higher)\n"
    "* = 95% CI excludes 1\n",
    fontsize=13, pad=50,
)

cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm_rr, cmap=cmap_rr), cax=cax)
rr_ticks = [1/2, 1/1.5, 1, 1.5, 2]
cb.set_ticks([np.log2(t) for t in rr_ticks])
cb.set_ticklabels([f"{t:.1f}" if t != 1 else "1.0" for t in rr_ticks])
cb.ax.tick_params(labelsize=8, length=0, pad=4)
cb.outline.set_visible(False)
cb.ax.text(0.5, 1.06, "P2 triaged\nhigher", ha="center", va="bottom", fontsize=9,
           color="#e74c3c", fontweight="bold", linespacing=1.4,
           transform=cb.ax.transAxes)
cb.ax.text(0.5, -0.06, "P1 triaged\nhigher", ha="center", va="top", fontsize=9,
           color="#3498db", fontweight="bold", linespacing=1.4,
           transform=cb.ax.transAxes)

plt.savefig(f"../figures/heatmap_risk_ratio{FIG_SUFFIX}.png", dpi=150, bbox_inches="tight")
plt.show()

takeaway: there seem to be differences, let us look at paired personae to understand whether they are significantly different. A paired approach is needed beause each persona sub-experiment has the same 1,164 patient, and so each patient should be individually compared, across personae

In [ ]:
# --- Pairwise differences — butterfly style ---
# Uses all_results, ordered_pairs, pair_names, y, n_p, title_y, group_starts
# computed in the data cell above.

metric_names = list(all_results.keys())

def _get_flipped(lookup, p1_key, p2_key):
    """Get -(P1-P2) = P2-P1 so positive means P2 has higher rate."""
    fwd_label = f"({PERSONA_FULL[p1_key]}, {PERSONA_FULL[p2_key]})"
    rev_label = f"({PERSONA_FULL[p2_key]}, {PERSONA_FULL[p1_key]})"
    if fwd_label in lookup:
        row = lookup[fwd_label]
        return -row["mean"], -row["hi"], -row["lo"]
    else:
        row = lookup[rev_label]
        return row["mean"], row["lo"], row["hi"]

lookups = {}
for mname in metric_names:
    res = all_results[mname]
    lookup = {}
    for _, row in res.iterrows():
        lookup[row["label"]] = row
    lookups[mname] = lookup

fig, axes = plt.subplots(2, 1, figsize=(14, 16))
fig.subplots_adjust(hspace=0.35, bottom=0.06)

for ax, mname in zip(axes, metric_names):
    lookup = lookups[mname]

    flat_data = []
    for p1_key, p2_key in ordered_pairs:
        m, lo, hi = _get_flipped(lookup, p1_key, p2_key)
        flat_data.append((m, lo, hi))

    all_vals = [v for m, lo, hi in flat_data for v in (m, lo, hi)]
    max_abs = max(abs(v) for v in all_vals) * 1.3

    for k in range(n_p):
        m, lo, hi = flat_data[k]
        if lo > 0:
            color = "#e74c3c"
        elif hi < 0:
            color = "#27ae60"
        else:
            color = "#bdc3c7"
        ax.barh(y[k], m, color=color, edgecolor="none", height=0.6, zorder=3)

    ax.set_xlim(-max_abs, max_abs)

    ax.axvline(0, color="white", linewidth=2, zorder=4)

    for k in range(n_p):
        m, lo, hi = flat_data[k]
        ax.errorbar(m, y[k], xerr=[[m - lo], [hi - m]],
                    fmt="none", ecolor="black", elinewidth=0.8, capsize=2, zorder=5)

    for k in range(n_p):
        m, lo, hi = flat_data[k]
        ci_lo_abs = min(abs(lo), abs(hi)) * 100
        ci_hi_abs = max(abs(lo), abs(hi)) * 100
        txt = f"{abs(m)*100:.1f}pp [{ci_lo_abs:.1f}, {ci_hi_abs:.1f}]"
        tip = hi if m >= 0 else lo
        pad = max_abs * 0.02
        if m >= 0:
            ax.text(tip + pad, y[k], txt, va="center", ha="left",
                    fontsize=8, color="black", zorder=6)
        else:
            ax.text(tip - pad, y[k], txt, va="center", ha="right",
                    fontsize=8, color="black", zorder=6)

    # Left y-axis: Persona 1
    left_labels = [p[0] for p in pair_names] + ["Persona 1"]
    left_ticks = list(y) + [title_y]
    ax.set_yticks(left_ticks)
    ax.set_yticklabels(left_labels, fontsize=10)
    ax.get_yticklabels()[-1].set_fontweight("bold")
    ax.set_ylim(-0.5, title_y + 0.5)

    # Right y-axis: Persona 2
    ax2 = ax.twinx()
    ax2.set_ylim(ax.get_ylim())
    right_labels = [p[1] for p in pair_names] + ["Persona 2"]
    ax2.set_yticks(left_ticks)
    ax2.set_yticklabels(right_labels, fontsize=10)
    ax2.get_yticklabels()[-1].set_fontweight("bold")
    ax2.spines[["top", "right", "left"]].set_visible(False)

    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.yaxis.set_tick_params(length=0)
    ax2.yaxis.set_tick_params(length=0)
    ax.xaxis.set_major_formatter(
        plt.FuncFormatter(lambda v, _: f"{abs(v)*100:.0f}pp" if v != 0 else "0"))
    ax.tick_params(axis="x", labelsize=9)

    # Row separators
    fig.canvas.draw()
    xlim = ax.get_xlim()
    for k in range(n_p - 1):
        mid = (y[k] + y[k + 1]) / 2
        is_group_boundary = (k + 1) in group_starts
        if is_group_boundary:
            ax.hlines(mid, xlim[0], xlim[1], color="grey", alpha=0.5,
                      lw=1.0, zorder=1)
        else:
            ax.hlines(mid, xlim[0], xlim[1], color="grey", alpha=0.2,
                      lw=0.5, zorder=0)

    # Directional labels anchored at center
    center_frac = (0 - xlim[0]) / (xlim[1] - xlim[0])
    verb = "over-triage" if "Over" in mname else "under-triage"
    ax.text(center_frac - 0.01, 1.005, f"← Persona 1 higher {verb} rate",
            transform=ax.transAxes, fontsize=11, ha="right", color="#27ae60")
    ax.text(center_frac + 0.01, 1.005, f"Persona 2 higher {verb} rate →",
            transform=ax.transAxes, fontsize=11, ha="left", color="#e74c3c")

    short = "over" if "Over" in mname else "under"
    ax.set_title(f"Same patient, different persona: who gets higher {short}-triage rate?",
                 fontsize=14, pad=40)
    ax.set_xlabel(f"{short.capitalize()}-triage rate absolute difference (pp)", fontsize=11)

plt.savefig(f"../figures/bar_comparison_butterfly{FIG_SUFFIX}.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Heatmap: pairwise over/under-triage rate differences (row − column) ---
# Triangular layout: 4 rows x 4 cols, all 10 unique pairs, no empty edges.
# Staircase fills from the left.

from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

persona_hm_order = [
    "dismissive_patient", "informed_advocate", "baseline",
    "limited_communicator", "anxious_patient",
]
persona_hm_labels = [
    "Dismissive\nPatient", "Informed\nAdvocate", "Default",
    "Limited\nCommunicator", "Anxious\nPatient",
]

# Rows = first 4, Cols = last 4 reversed → staircase fills left
row_personas = persona_hm_order[:-1]
col_personas = persona_hm_order[1:][::-1]
row_labels = persona_hm_labels[:-1]
col_labels = persona_hm_labels[1:][::-1]
n_rows = len(row_personas)
n_cols = len(col_personas)

# Reversed colormap: positive (P1 higher rate = worse) is red, negative is green
cmap = LinearSegmentedColormap.from_list("GnWtRd", ["#57bb8a", "#ffffff", "#e67c73"])
cmap.set_bad("#f5f5f5")

fig, axes = plt.subplots(
    2, 2, figsize=(16, 8),
    gridspec_kw={"hspace": 0.65, "width_ratios": [1, 0.02], "wspace": 0.02},
)

for row_idx, mname in enumerate(metric_names):
    ax = axes[row_idx, 0]
    cax = axes[row_idx, 1]
    lookup = lookups[mname]

    matrix = np.full((n_rows, n_cols), np.nan)
    annot_txt = [[""] * n_cols for _ in range(n_rows)]

    for ri, p_row in enumerate(row_personas):
        for ci, p_col in enumerate(col_personas):
            idx_row = persona_hm_order.index(p_row)
            idx_col = persona_hm_order.index(p_col)
            if idx_col <= idx_row:
                continue
            m, lo, hi = _get_flipped(lookup, p_col, p_row)
            matrix[ri, ci] = m
            annot_txt[ri][ci] = f"{m * 100:+.1f}  ({lo * 100:.1f}, {hi * 100:.1f})"

    masked = np.ma.masked_invalid(matrix)
    vmax = np.nanmax(np.abs(matrix))
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

    ax.imshow(masked, cmap=cmap, norm=norm, aspect="auto")

    for i in range(n_rows):
        for j in range(n_cols):
            if annot_txt[i][j]:
                ax.text(j, i, annot_txt[i][j], ha="center", va="center",
                        fontsize=11, color="black")

    for edge in range(max(n_rows, n_cols) + 1):
        if edge <= n_rows:
            ax.axhline(edge - 0.5, color="white", lw=2)
        if edge <= n_cols:
            ax.axvline(edge - 0.5, color="white", lw=2)

    # Corner label — top-left
    ax.text(-0.5, -1.25, "Persona 2 →\nPersona 1 ↓", ha="right", va="center",
            fontsize=12, color="black", linespacing=1.5,
            transform=ax.transData)

    # Column labels (top)
    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(col_labels, fontsize=12, linespacing=1.1)
    ax.xaxis.tick_top()
    ax.tick_params(axis="x", pad=12)

    # Row labels (left)
    ax.set_yticks(range(n_rows))
    ax.set_yticklabels(row_labels, fontsize=12, linespacing=1.1)
    ax.tick_params(axis="y", pad=12)

    ax.tick_params(length=0)
    ax.spines[["top", "right", "bottom", "left"]].set_visible(False)

    short = "over" if "Over" in mname else "under"
    ax.set_title(
        f"Pairwise {short}-triage rate difference (in percentage point, pp)\n",
        fontsize=13, pad=50,
    )

    # Colorbar
    cb = fig.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cax,
    )
    cb.set_ticks([-vmax, 0, vmax])
    cb.set_ticklabels([f"{-vmax * 100:.0f}", "0", f"+{vmax * 100:.0f}"])
    cb.ax.tick_params(labelsize=8, length=0, pad=4)
    cb.outline.set_visible(False)
    cb.ax.text(0.5, 1.06, "P1 > P2\nP1 worse", ha="center", va="bottom", fontsize=9,
               color="#e67c73", fontweight="bold", linespacing=1.4,
               transform=cb.ax.transAxes)
    cb.ax.text(0.5, -0.06, "P1 < P2\nP1 better", ha="center", va="top", fontsize=9,
               color="#57bb8a", fontweight="bold", linespacing=1.4,
               transform=cb.ax.transAxes)

plt.savefig(f"../figures/heatmap_pairwise{FIG_SUFFIX}.png", dpi=150, bbox_inches="tight")
plt.show()

## Demographics breakdown

Error rates by demographic subgroup (sex, age) across personae.

In [ ]:
# --- Demographics breakdown (all personas) ---
df["gender_d"] = df["gender"].map({"Male": "M", "Female": "F"}).fillna(df["gender"])
df["age_group"] = pd.cut(
    pd.to_numeric(df["age"], errors="coerce"),
    bins=[0, 30, 50, 70, 100],
    labels=["<=30", "31-50", "51-70", "71+"],
)


def group_metrics(grp):
    ot_eligible = grp[grp["gold_triage"] != "urgent"]
    over_vals = (ot_eligible["direction"] == "over").astype(int).tolist() if len(ot_eligible) else [0]

    ut_eligible = grp[grp["gold_triage"] != "self-care"]
    under_vals = (ut_eligible["direction"] == "under").astype(int).tolist() if len(ut_eligible) else [0]

    o_mean, o_lo, o_hi = bootstrap_rate(over_vals)
    u_mean, u_lo, u_hi = bootstrap_rate(under_vals)

    return pd.Series({
        "n": len(grp),
        "over_triage": f"{o_mean:.1%} [{o_lo:.1%}, {o_hi:.1%}]",
        "under_triage": f"{u_mean:.1%} [{u_lo:.1%}, {u_hi:.1%}]",
    })


print("\n=== By persona x sex ===")
display(df.groupby(["condition_name", "gender_d"]).apply(group_metrics).reset_index())

print("\n=== By persona x age group ===")
display(df.groupby(["condition_name", "age_group"], observed=True).apply(group_metrics).reset_index())

In [ ]:
# --- Persona x demographics grouped bar plots ---

df["gender_clean"] = df["gender"].map({"Male": "M", "Female": "F"}).fillna(df["gender"])
df["age_numeric"] = pd.to_numeric(df["age"], errors="coerce")
df["age_grp"] = pd.cut(
    df["age_numeric"], bins=[0, 30, 50, 70, 100], labels=["<=30", "31-50", "51-70", "71+"]
)

rng_dem = np.random.default_rng(42)

def _boot_rate_dem(vals, rng, n_boot=10_000):
    arr = np.array(vals, dtype=float)
    if len(arr) == 0:
        return 0.0, 0.0, 0.0
    boots = np.array([rng.choice(arr, size=len(arr), replace=True).mean() for _ in range(n_boot)])
    return float(arr.mean()), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

persona_order = sorted(df["condition_name"].unique())

dem_metrics = [
    ("Over-triage rate", lambda d: d[d["gold_triage"] != "urgent"], "over"),
    ("Under-triage rate", lambda d: d[d["gold_triage"] != "self-care"], "under"),
]

sexes = ["F", "M"]
sex_colors = {"F": "#e74c3c", "M": "#3498db"}
age_groups = ["<=30", "31-50", "51-70", "71+"]
age_cmap = ["#bdd7e7", "#6baed6", "#3182bd", "#08519c"]

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharey="row")

for row_idx, (mname, filt_fn, dval) in enumerate(dem_metrics):
    x = np.arange(len(persona_order))

    ax = axes[row_idx, 0]
    bw = 0.35
    for s_idx, sex in enumerate(sexes):
        ms, los, his, ns = [], [], [], []
        for persona in persona_order:
            sub = filt_fn(df[(df["condition_name"] == persona) & (df["gender_clean"] == sex)])
            vals = (sub["direction"] == dval).astype(int).values
            m, lo, hi = _boot_rate_dem(vals, rng_dem)
            ms.append(m); los.append(lo); his.append(hi); ns.append(len(vals))
        ms, los, his = np.array(ms), np.array(los), np.array(his)
        offset = (s_idx - 0.5) * bw
        ax.bar(x + offset, ms, bw, edgecolor="none", color=sex_colors[sex], label=sex, zorder=3)
        ax.errorbar(x + offset, ms, yerr=[ms - los, his - ms],
                    fmt="none", ecolor="black", elinewidth=0.8, capsize=2, zorder=4)
        for p_idx in range(len(persona_order)):
            ax.text(p_idx + offset, 0.005, f"n={ns[p_idx]}",
                    ha="center", va="bottom", fontsize=6, color="white")
    ax.set_xticks(x)
    ax.set_xticklabels([PERSONA_FULL.get(p, p) for p in persona_order], fontsize=9)
    ax.set_title(f"{mname} by sex", fontsize=11)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3, zorder=0)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.set_ylabel(mname, fontsize=10)
    ax.legend(fontsize=9, loc="upper right")

    ax = axes[row_idx, 1]
    n_ag = len(age_groups)
    bw_a = 0.8 / n_ag
    for g_idx, ag in enumerate(age_groups):
        ms, los, his, ns = [], [], [], []
        for persona in persona_order:
            sub = filt_fn(df[(df["condition_name"] == persona) & (df["age_grp"] == ag)])
            vals = (sub["direction"] == dval).astype(int).values
            m, lo, hi = _boot_rate_dem(vals, rng_dem)
            ms.append(m); los.append(lo); his.append(hi); ns.append(len(vals))
        ms, los, his = np.array(ms), np.array(los), np.array(his)
        offset = (g_idx - (n_ag - 1) / 2) * bw_a
        ax.bar(x + offset, ms, bw_a, edgecolor="none", color=age_cmap[g_idx], label=ag, zorder=3)
        ax.errorbar(x + offset, ms, yerr=[ms - los, his - ms],
                    fmt="none", ecolor="black", elinewidth=0.8, capsize=2, zorder=4)
        for p_idx in range(len(persona_order)):
            ax.text(p_idx + offset, 0.003, f"n={ns[p_idx]}",
                    ha="center", va="bottom", fontsize=5, color="white")
    ax.set_xticks(x)
    ax.set_xticklabels([PERSONA_FULL.get(p, p) for p in persona_order], fontsize=9)
    ax.set_title(f"{mname} by age group", fontsize=11)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3, zorder=0)
    ax.legend(fontsize=8, ncol=2, loc="upper right")

fig.suptitle("Error Rates by Demographic subgroup, across 5 Personae", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f"../figures/bar_demographics{FIG_SUFFIX}.png", dpi=150, bbox_inches="tight")
plt.show()

takeaway: no significant demographics differences. the variation across personae is significant though!

## Probability-based analyses

The following analyses use estimated probabilities from multiple assessment runs (different seeds), providing a continuous measure of triage severity rather than the discrete majority-vote prediction. This is a surrogate to the estimated probabilities that a model would typically output. The c-stat and calibration-in-the-large are an attempt to understand the drivers of the differences that we verified above but must be interpreted with caution.

In [ ]:
# --- AUC from assessment probabilities ---
from sklearn.metrics import roc_auc_score

def _assessment_probs(assessments):
    if not isinstance(assessments, list) or not assessments:
        return {"self-care": np.nan, "non-urgent": np.nan, "urgent": np.nan}
    votes = [a["urgency"] for a in assessments if a.get("urgency")]
    n = len(votes)
    if n == 0:
        return {"self-care": np.nan, "non-urgent": np.nan, "urgent": np.nan}
    counts = Counter(votes)
    return {k: counts.get(k, 0) / n for k in ["self-care", "non-urgent", "urgent"]}

probs_list = df["assessments"].apply(_assessment_probs).tolist()
probs_df = pd.DataFrame(probs_list)
df["p_selfcare"] = probs_df["self-care"].values
df["p_nonurgent"] = probs_df["non-urgent"].values
df["p_urgent"] = probs_df["urgent"].values

print(f"Assessment probability distributions (mean across all cases):")
print(f"  P(self-care):  {df['p_selfcare'].mean():.3f}")
print(f"  P(non-urgent): {df['p_nonurgent'].mean():.3f}")
print(f"  P(urgent):     {df['p_urgent'].mean():.3f}")


def bootstrap_auc(y_true, y_score, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    arr_y = np.array(y_true, dtype=int)
    arr_s = np.array(y_score, dtype=float)
    n = len(arr_y)
    base = roc_auc_score(arr_y, arr_s)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt = arr_y[idx]
        if yt.sum() == 0 or yt.sum() == n:
            continue
        boots.append(roc_auc_score(yt, arr_s[idx]))
    return base, np.percentile(boots, 2.5), np.percentile(boots, 97.5)


auc_pairs = [
    ("non-urgent", "self-care", "p_nonurgent", "C-stat of P(Non-Urgent) vs P(Self-Care)"),
    ("urgent", "non-urgent", "p_urgent", "C-stat of P(Urgent) vs P(Non-Urgent)"),
    ("urgent", "self-care", "p_urgent", "C-stat of P(Urgent) vs P(Self-Care)"),
]

auc_rows = []
for cond in sorted(df["condition_name"].unique()):
    cond_df = df[df["condition_name"] == cond]
    for pos_class, neg_class, score_col, label in auc_pairs:
        subset = cond_df[cond_df["gold_triage"].isin([pos_class, neg_class])]
        subset = subset[subset[score_col].notna()]
        y_true = (subset["gold_triage"] == pos_class).astype(int).values
        y_score = subset[score_col].values
        if len(np.unique(y_true)) < 2 or len(subset) < 10:
            continue
        auc, lo, hi = bootstrap_auc(y_true, y_score)
        auc_rows.append({
            "condition": cond, "comparison": label,
            "auc_mean": auc, "auc_lo": lo, "auc_hi": hi,
            "n": len(subset),
        })

auc_df = pd.DataFrame(auc_rows)
disp = auc_df.copy()
disp["AUC"] = disp.apply(lambda r: f"{r.auc_mean:.3f} [{r.auc_lo:.3f}, {r.auc_hi:.3f}]", axis=1)
print("\n=== One-vs-one AUC by condition (from assessment probabilities) ===")
display(disp.pivot(index="condition", columns="comparison", values="AUC"))

In [ ]:
# Bar plot for AUC
comparisons = [c for _, _, _, c in auc_pairs]

cond_order = sorted(df["condition_name"].unique())
y_labels = [PERSONA_FULL.get(c, c) for c in cond_order]
y_pos = np.arange(len(cond_order))

fig, axes = plt.subplots(3, 1, figsize=(9, 14), sharey=True)
fig.subplots_adjust(hspace=0.35, top=0.94)

for ax, comp in zip(axes, comparisons):
    comp_data = auc_df[auc_df["comparison"] == comp].set_index("condition").reindex(cond_order)
    vals = comp_data["auc_mean"].values
    lo = comp_data["auc_lo"].values
    hi = comp_data["auc_hi"].values

    ax.barh(y_pos, vals, color="#27ae60", edgecolor="none", height=0.5, zorder=3)
    ax.errorbar(vals, y_pos, xerr=[vals - lo, hi - vals],
                fmt="none", ecolor="black", elinewidth=0.8, capsize=2, zorder=4)

    for k in range(len(y_pos)):
        txt = f"{vals[k]:.3f} [{lo[k]:.3f}, {hi[k]:.3f}]"
        ax.annotate(txt, xy=(1.0, y_pos[k]), xycoords=("axes fraction", "data"),
                    xytext=(10, 0), textcoords="offset points",
                    va="center", ha="left", fontsize=7, color="black",
                    annotation_clip=False)

    ax.set_xlim(0.5, 1.05)
    ax.set_title(comp, fontsize=11)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(alpha=0.3, zorder=0)
    ax.tick_params(labelsize=8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(y_labels, fontsize=9)

fig.suptitle("Overall Discrimination across Personae", fontsize=13)
plt.savefig(f"../figures/auc_bar{FIG_SUFFIX}.png", dpi=150, bbox_inches="tight")
plt.show()

takeaway: it doesn't look like there are big/significant differences in discrimination across the different personae but that does not preclude under- and over-triage rate differences. so it has to be a calibration issue. calibration-in-the-large can quickly show that, and so does the last plot with mean estimated probablities.

In [ ]:
# --- Pairwise probability differences (paired bootstrap CIs) ---
# Same ordered_pairs grouping as the pairwise differences butterfly.

prob_metrics = [
    ("Difference in P(self-care) across Pairs", "p_selfcare"),
    ("Difference in P(non-urgent) across Pairs", "p_nonurgent"),
    ("Difference in P(urgent) across Pairs", "p_urgent"),
]

# Precompute all pairwise differences for each metric
prob_lookups = {}
for name, col in prob_metrics:
    rng = np.random.default_rng(42)
    piv = df.pivot_table(
        index=["case_id", "repetition"], columns="condition_name",
        values=col, aggfunc="mean",
    ).dropna()

    lookup = {}
    for i in range(len(personas)):
        for j in range(i + 1, len(personas)):
            d = (piv[personas[i]] - piv[personas[j]]).values
            boots = np.array([
                rng.choice(d, size=len(d), replace=True).mean() for _ in range(10_000)
            ])
            lo, hi = np.percentile(boots, [2.5, 97.5])
            label = f"({PERSONA_FULL[personas[i]]}, {PERSONA_FULL[personas[j]]})"
            lookup[label] = {"mean": d.mean(), "lo": lo, "hi": hi}
    prob_lookups[name] = lookup

fig, axes = plt.subplots(3, 1, figsize=(9, 18))
fig.subplots_adjust(hspace=0.25, top=0.95)

for ax, (name, col) in zip(axes, prob_metrics):
    lookup = prob_lookups[name]

    flat_data = []
    for p1_key, p2_key in ordered_pairs:
        fwd = f"({PERSONA_FULL[p1_key]}, {PERSONA_FULL[p2_key]})"
        rev = f"({PERSONA_FULL[p2_key]}, {PERSONA_FULL[p1_key]})"
        if fwd in lookup:
            r = lookup[fwd]
            m, lo, hi = -r["mean"], -r["hi"], -r["lo"]
        else:
            r = lookup[rev]
            m, lo, hi = r["mean"], r["lo"], r["hi"]
        flat_data.append((m, lo, hi))

    all_vals = [v for m, lo, hi in flat_data for v in (m, lo, hi)]
    max_abs = max(abs(v) for v in all_vals) * 1.5

    for k in range(n_p):
        m, lo, hi = flat_data[k]
        if lo > 0:
            color = "#e74c3c"
        elif hi < 0:
            color = "#27ae60"
        else:
            color = "#bdc3c7"
        ax.barh(y[k], m, color=color, edgecolor="none", height=0.6, zorder=3)

    ax.set_xlim(-max_abs, max_abs)
    ax.axvline(0, color="white", linewidth=2, zorder=4)

    for k in range(n_p):
        m, lo, hi = flat_data[k]
        ax.errorbar(m, y[k], xerr=[[m - lo], [hi - m]],
                    fmt="none", ecolor="black", elinewidth=0.8, capsize=2, zorder=5)

    for k in range(n_p):
        m, lo, hi = flat_data[k]
        ci_lo_abs = min(abs(lo), abs(hi)) * 100
        ci_hi_abs = max(abs(lo), abs(hi)) * 100
        txt = f"{abs(m)*100:.1f}pp [{ci_lo_abs:.1f}, {ci_hi_abs:.1f}]"
        tip = hi if m >= 0 else lo
        pad = max_abs * 0.02
        if m >= 0:
            ax.text(tip + pad, y[k], txt, va="center", ha="left",
                    fontsize=8, color="black", zorder=6)
        else:
            ax.text(tip - pad, y[k], txt, va="center", ha="right",
                    fontsize=8, color="black", zorder=6)

    # Left y-axis: Persona 1
    left_labels = [p[0] for p in pair_names] + ["Persona 1"]
    left_ticks = list(y) + [title_y]
    ax.set_yticks(left_ticks)
    ax.set_yticklabels(left_labels, fontsize=10)
    ax.get_yticklabels()[-1].set_fontweight("bold")
    ax.set_ylim(-0.5, title_y + 0.5)

    # Right y-axis: Persona 2
    ax2 = ax.twinx()
    ax2.set_ylim(ax.get_ylim())
    right_labels = [p[1] for p in pair_names] + ["Persona 2"]
    ax2.set_yticks(left_ticks)
    ax2.set_yticklabels(right_labels, fontsize=10)
    ax2.get_yticklabels()[-1].set_fontweight("bold")
    ax2.spines[["top", "right", "left"]].set_visible(False)

    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.yaxis.set_tick_params(length=0)
    ax2.yaxis.set_tick_params(length=0)
    ax.xaxis.set_major_formatter(
        plt.FuncFormatter(lambda v, _: f"{abs(v)*100:.0f}pp" if v != 0 else "0"))
    ax.tick_params(axis="x", labelsize=9)

    ax.set_title(name, fontsize=12, pad=10)

    # Row separators
    fig.canvas.draw()
    xlim = ax.get_xlim()
    for k in range(n_p - 1):
        mid = (y[k] + y[k + 1]) / 2
        is_group_boundary = (k + 1) in group_starts
        if is_group_boundary:
            ax.hlines(mid, xlim[0], xlim[1], color="grey", alpha=0.5,
                      lw=1.0, zorder=1)
        else:
            ax.hlines(mid, xlim[0], xlim[1], color="grey", alpha=0.2,
                      lw=0.5, zorder=0)

fig.suptitle("Variation of Mean Estimated Probability across Personae, clustered by Patient",
             fontsize=13)
plt.savefig(f"../figures/bar_prob_comparison{FIG_SUFFIX}.png", dpi=150, bbox_inches="tight")
plt.show()

this one was trivial: there are difference in estimated probabilities across personae

In [ ]:
# --- Calibration in the large ---

triage_levels = ["self-care", "non-urgent", "urgent"]
triage_titles = ["Calibration-in-the-large of P(Self-Care)",
                 "Calibration-in-the-large of P(Non-Urgent)",
                 "Calibration-in-the-large of P(Urgent)"]
prob_cols = ["p_selfcare", "p_nonurgent", "p_urgent"]

cond_order = sorted(df["condition_name"].unique())
y_labels = [PERSONA_FULL.get(c, c) for c in cond_order]
y_pos = np.arange(len(cond_order))

cal_rows = []
for cond in cond_order:
    sub = df[df["condition_name"] == cond]
    for level, pcol in zip(triage_levels, prob_cols):
        pred = sub[pcol].values
        obs = (sub["gold_triage"] == level).astype(float).values
        diff = pred - obs
        mean_cal = diff.mean()
        rng = np.random.default_rng(42)
        boots = np.array([rng.choice(diff, size=len(diff), replace=True).mean()
                          for _ in range(10_000)])
        lo, hi = np.percentile(boots, [2.5, 97.5])
        cal_rows.append({"condition": cond, "triage_level": level,
                         "cal": mean_cal, "lo": lo, "hi": hi})

cal_df = pd.DataFrame(cal_rows)

fig, axes = plt.subplots(3, 1, figsize=(9, 13))
fig.subplots_adjust(hspace=0.30, top=0.94, bottom=0.06)

for idx_ax, (ax, level, title) in enumerate(zip(axes, triage_levels, triage_titles)):
    level_data = cal_df[cal_df["triage_level"] == level].set_index("condition").reindex(cond_order)
    vals = level_data["cal"].values
    lo = level_data["lo"].values
    hi = level_data["hi"].values

    colors = []
    for l, h in zip(lo, hi):
        if l > 0:
            colors.append("#27ae60")
        elif h < 0:
            colors.append("#e74c3c")
        else:
            colors.append("#bdc3c7")

    ax.barh(y_pos, vals, color=colors, edgecolor="none", height=0.5, zorder=3)
    ax.errorbar(vals, y_pos, xerr=[vals - lo, hi - vals],
                fmt="none", ecolor="black", elinewidth=0.8, capsize=2, zorder=4)
    ax.axvline(0, color="black", linewidth=0.8, zorder=2)

    for k in range(len(y_pos)):
        txt = f"{vals[k]*100:+.1f}pp [{lo[k]*100:+.1f}, {hi[k]*100:+.1f}]"
        ax.annotate(txt, xy=(1.0, y_pos[k]), xycoords=("axes fraction", "data"),
                    xytext=(10, 0), textcoords="offset points",
                    va="center", ha="left", fontsize=7, color="black",
                    annotation_clip=False)

    ax.set_title(title, fontsize=11)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(alpha=0.3, zorder=0)
    ax.xaxis.set_major_formatter(
        plt.FuncFormatter(lambda v, _: f"{v*100:+.0f}pp" if v != 0 else "0pp"))
    ax.tick_params(labelsize=8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(y_labels, fontsize=9)

from matplotlib.patches import Patch
legend_handles = [
    Patch(facecolor="#27ae60", edgecolor="none",
          label="Over-estimated"),
    Patch(facecolor="#bdc3c7", edgecolor="none",
          label="No significant miscalibration"),
    Patch(facecolor="#e74c3c", edgecolor="none",
          label="Under-estimated"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=3, fontsize=9,
           frameon=False, bbox_to_anchor=(0.5, 0.0))

fig.suptitle("Overall Calibration across Personae", fontsize=13)
plt.savefig(f"../figures/calibration_large{FIG_SUFFIX}.png", dpi=150, bbox_inches="tight")
plt.show()

takeaway: it's mostly a calibration issue indeed